In [1]:
from pathlib import Path

import pyarrow.parquet as pq

work_dir = Path.cwd().resolve()
repo_root = work_dir.parent if work_dir.name == "notebooks" else work_dir
parquet_path = repo_root / (
    "data/verified_order_exports/binance-intra-arb01/"
    "full_verified_through_20260731T090747Z/"
    "20260715T133700.000000Z__20260731T090747.006999Z/"
    "uniform_orders.parquet"
)

if not parquet_path.is_file():
    raise FileNotFoundError(parquet_path)

parquet_file = pq.ParquetFile(parquet_path)
print(f"file: {parquet_path}")
print(f"rows: {parquet_file.metadata.num_rows:,}")
print(f"row groups: {parquet_file.metadata.num_row_groups}")


file: /home/ubuntu/crypto_nav_manager/data/verified_order_exports/binance-intra-arb01/full_verified_through_20260731T090747Z/20260715T133700.000000Z__20260731T090747.006999Z/uniform_orders.parquet
rows: 1,574,106
row groups: 6


In [2]:
excluded_columns = {"key", "recv_ts_us", "from_key_hex"}
schema = parquet_file.schema_arrow
selected_fields = [field for field in schema if field.name not in excluded_columns]
selected_columns = [field.name for field in selected_fields]
print(f"columns: {len(selected_fields)}")
for index, field in enumerate(selected_fields, start=1):
    print(f"{index:>2}. {field.name:<20} {field.type}")


columns: 19
 1. ts_us                int64
 2. symbol               large_string
 3. create_ts            int64
 4. update_ts            int64
 5. signal_ts            int64
 6. submit_ts            int64
 7. local_ts             int64
 8. mkt_ts               int64
 9. client_order_id      int64
10. trading_venue        large_string
11. order_type           large_string
12. side                 large_string
13. price                double
14. price_offset         double
15. amount_init          double
16. amount_update        double
17. status               large_string
18. from_key             large_string
19. bbo_spread           large_string


In [3]:
head = next(
    parquet_file.iter_batches(batch_size=5, columns=selected_columns)
).to_pandas()
head.head(5)


,ts_us,symbol,create_ts,update_ts,signal_ts,submit_ts,local_ts,mkt_ts,client_order_id,trading_venue,order_type,side,price,price_offset,amount_init,amount_update,status,from_key,bbo_spread
0,1784122740704960,XRPUSDT,1784122725000821,1784122740704948,1784122725000737,1784122732684942,1784122740704949,1784122724999863,8161197977122111489,BinanceMargin,LIMIT,SELL,1.1274,0.000089,44.300,0.0,EXPIRED,8161197977122111489|1784122725000737:ret_qtl=0...,"1784122740661532,1.1276,25825.5,1.1277,20034.2..."
1,1784122740964953,ETHUSDT,1784122725002386,1784122740964948,1784122725002365,1784122732944942,1784122740964948,1784122725249135,8161197985712046081,BinanceMargin,LIMIT,BUY,1938.1300,0.000000,0.025,0.0,EXPIRED,8161197985712046081|1784122725002365:ret_qtl=0...,"1784122740959865,1937.04,3.6165,1937.05,102.88..."
2,1784122740964967,ETHUSDT,1784122725002393,1784122740964964,1784122725002365,1784122732944947,1784122740964964,1784122725249135,8161197990007013377,BinanceMargin,LIMIT,BUY,1937.5500,0.000299,0.025,0.0,EXPIRED,8161197990007013377|1784122725002365:ret_qtl=0...,"1784122740959865,1937.04,3.6165,1937.05,102.88..."
3,1784122740964979,SUIUSDT,1784122725179705,1784122740964975,1784122725179676,1784122732944950,1784122740964975,1784122725262471,8161198037251653633,BinanceMargin,LIMIT,SELL,0.7672,0.000391,65.100,0.0,EXPIRED,8161198037251653633|1784122725179676:ret_qtl=0...,"1784122740960112,0.7665,1167.8,0.7666,3217.6,1..."
4,1784122740964994,SUIUSDT,1784122725179698,1784122740964990,1784122725179676,1784122732944952,1784122740964990,1784122725262471,8161198032956686337,BinanceMargin,LIMIT,SELL,0.7669,0.000000,65.100,0.0,EXPIRED,8161198032956686337|1784122725179676:ret_qtl=0...,"1784122740960112,0.7665,1167.8,0.7666,3217.6,1..."


In [4]:
margin_fills_table = pq.read_table(
    parquet_path,
    columns=selected_columns,
    filters=[
        ("trading_venue", "==", "BinanceMargin"),
        ("amount_update", ">", 0.0),
    ],
)
margin_fills = margin_fills_table.to_pandas()
print(f"Margin fill rows: {len(margin_fills):,}")
print(f"Margin orders: {margin_fills.client_order_id.nunique():,}")
print(f"Margin filled quantity: {margin_fills.amount_update.sum():,.8f}")
margin_fills.head(5)


Margin fill rows: 30,497
Margin orders: 27,736
Margin filled quantity: 21,118,460.48840000


,ts_us,symbol,create_ts,update_ts,signal_ts,submit_ts,local_ts,mkt_ts,client_order_id,trading_venue,order_type,side,price,price_offset,amount_init,amount_update,status,from_key,bbo_spread
0,1784123021403859,XRPUSDT,1784123020486682,1784123021374000,1784123020486658,1784123021374139,1784123021374771,1784123021373333,183663753743564801,BinanceMargin,LIMIT,SELL,1.1268,0.000000e+00,44.300,44.300,FILLED,183663753743564801|1784123020486658:ret_qtl=0:...,"1784123021373333,1.1267,21655.5,1.1268,11515.1..."
1,1784123022531030,ETHUSDT,1784123021965759,1784123022530000,1784123021965708,1784123021965759,1784123022531020,1784123021965303,183663770923433985,BinanceMargin,LIMIT,BUY,1934.4700,0.000000e+00,0.025,0.025,FILLED,183663770923433985|1784123021965708:ret_qtl=0:...,"1784123022405828,1934.47,25.6927,1934.48,29.60..."
2,1784123026319704,ZAMAUSDT,1784123015547674,1784123026319000,1784123015547652,1784123026318842,1784123026319678,1784123026317000,183663680729120769,BinanceMargin,LIMIT,SELL,0.0341,2.034866e-16,1466.000,1466.000,FILLED,183663680729120769|1784123015547652:ret_qtl=0:...,"1784123020843279,0.03409,8656,0.0341,15283,178..."
3,1784123026319727,ZAMAUSDT,1784123020842094,1784123026319000,1784123020842073,1784123026318846,1784123026319720,1784123026317000,183663762333499393,BinanceMargin,LIMIT,SELL,0.0341,2.034866e-16,1466.000,1466.000,FILLED,183663762333499393|1784123020842073:ret_qtl=0:...,"1784123020843279,0.03409,8656,0.0341,15283,178..."
4,1784123026937489,SOLUSDT,1784123022837528,1784123026937000,1784123022837512,1784123022837528,1784123026937468,1784123022836000,183663800988205057,BinanceMargin,LIMIT,SELL,78.5600,0.000000e+00,0.630,0.630,FILLED,183663800988205057|1784123022837512:ret_qtl=0:...,"1784123026706720,78.55,152.663,78.56,69.123,17..."


In [5]:
lifecycle_columns = [
    "ts_us",
    "client_order_id",
    "trading_venue",
    "create_ts",
    "update_ts",
    "amount_init",
    "amount_update",
    "price",
    "price_offset",
    "from_key",
    "symbol",
    "side",
]

margin_events = pq.read_table(
    parquet_path,
    columns=lifecycle_columns,
    filters=[("trading_venue", "==", "BinanceMargin")],
).to_pandas()
open_client_ids = set(margin_fills.client_order_id.astype("int64"))
open_events = margin_events[
    margin_events.client_order_id.isin(open_client_ids)
].copy()
open_last = (
    open_events.sort_values(["client_order_id", "update_ts", "ts_us"])
    .groupby("client_order_id", as_index=False)
    .tail(1)
)
open_summary = (
    open_events.groupby("client_order_id", as_index=False)
    .agg(cts=("create_ts", "min"), open_uts=("update_ts", "max"))
    .merge(
        open_last[
            [
                "client_order_id",
                "symbol",
                "side",
                "price",
                "amount_init",
                "price_offset",
                "from_key",
            ]
        ],
        on="client_order_id",
        how="left",
        validate="one_to_one",
    )
    .rename(columns={"client_order_id": "fkey", "amount_init": "amount"})
)
open_summary["range"] = (open_summary.price_offset * 10_000).round(8)
open_summary["tlen"] = (
    open_summary.from_key.str.extract(r":tlen=([^:]+)$", expand=False).astype("float64") * open_summary.price
)
if open_summary.tlen.isna().any():
    raise ValueError("A BinanceMargin from_key is missing terminal tlen")
open_summary = open_summary.drop(columns=["price_offset", "from_key"])

futures_events = pq.read_table(
    parquet_path,
    columns=lifecycle_columns,
    filters=[("trading_venue", "==", "BinanceFutures")],
).to_pandas()
fkey_text = futures_events.from_key.str.partition("|")[0]
if not fkey_text.str.fullmatch(r"[0-9]+").all():
    raise ValueError("A BinanceFutures from_key has a non-numeric prefix")
futures_events["fkey"] = fkey_text.astype("int64")
futures_events["weighted_close_value"] = (
    futures_events.price * futures_events.amount_update
)
futures_summary = (
    futures_events.groupby("fkey", as_index=False)
    .agg(
        fts=("update_ts", "max"),
        close_count=("update_ts", "size"),
        camount=("amount_update", "sum"),
        weighted_close_value=("weighted_close_value", "sum"),
    )
)
futures_summary["cprice"] = (
    futures_summary.weighted_close_value
    / futures_summary.camount.where(futures_summary.camount > 0)
)
futures_summary = futures_summary.drop(columns="weighted_close_value")

intra_orders = open_summary.merge(
    futures_summary,
    on="fkey",
    how="left",
    validate="one_to_one",
)
intra_orders["fts"] = intra_orders.fts.astype("Int64")
intra_orders["holding"] = intra_orders.open_uts - intra_orders.cts
intra_orders["holding_close"] = intra_orders.fts - intra_orders.open_uts
intra_orders["close_count"] = intra_orders.close_count.fillna(0).astype("int64")
intra_orders["camount"] = intra_orders.camount.fillna(0.0)
intra_orders["crange"] = -1.0
intra_orders["pnlu"] = float("nan")
buy_mask = intra_orders.side.str.lower().eq("buy")
sell_mask = intra_orders.side.str.lower().eq("sell")
intra_orders.loc[buy_mask, "pnlu"] = (
    (intra_orders.loc[buy_mask, "cprice"] - intra_orders.loc[buy_mask, "price"])
    / intra_orders.loc[buy_mask, "cprice"]
)
intra_orders.loc[sell_mask, "pnlu"] = (
    (intra_orders.loc[sell_mask, "price"] - intra_orders.loc[sell_mask, "cprice"])
    / intra_orders.loc[sell_mask, "price"]
)
intra_orders = intra_orders[
    [
        "fkey",
        "symbol",
        "side",
        "cts",
        "open_uts",
        "fts",
        "holding",
        "holding_close",
        "close_count",
        "price",
        "amount",
        "cprice",
        "camount",
        "range",
        "crange",
        "tlen",
        "pnlu",
    ]
]
matched_intra_orders = intra_orders.dropna(subset=["fts"]).copy()

print(f"Real Margin open orders: {len(intra_orders):,}")
print(f"Mapped to Futures fkey: {len(matched_intra_orders):,}")
print(f"Missing Futures fkey: {intra_orders.fts.isna().sum():,}")
print(f"Negative holding: {(intra_orders.holding < 0).sum():,}")
print(f"Negative holding_close: {(intra_orders.holding_close < 0).sum():,}")
print(f"crange=-1 rows: {(intra_orders.crange == -1).sum():,}")
print(f"Parsed tlen rows: {intra_orders.tlen.notna().sum():,}")
print(f"Computed pnlu rows: {intra_orders.pnlu.notna().sum():,}")
intra_orders.head(5)

Real Margin open orders: 27,736
Mapped to Futures fkey: 26,388
Missing Futures fkey: 1,348
Negative holding: 2
Negative holding_close: 0
crange=-1 rows: 27,736
Parsed tlen rows: 27,736
Computed pnlu rows: 26,388


,fkey,symbol,side,cts,open_uts,fts,holding,holding_close,close_count,price,amount,cprice,camount,range,crange,tlen,pnlu
0,183663680729120769,ZAMAUSDT,SELL,1784123015547674,1784123026319000,1784123026321000,10771326,2000,2,0.0341,1466.000,0.034040,1466.000,0.0,-1.0,421.169100,0.001760
1,183663753743564801,XRPUSDT,SELL,1784123020486682,1784123021374000,1784123022226000,887318,852000,2,1.1268,44.300,1.126200,44.300,0.0,-1.0,15453.385920,0.000532
2,183663762333499393,ZAMAUSDT,SELL,1784123020842094,1784123026319000,1784123026321000,5476906,2000,7,0.0341,1466.000,0.034038,1466.000,0.0,-1.0,471.159700,0.001819
3,183663770923433985,ETHUSDT,BUY,1784123021965759,1784123022530000,1784123025208000,564241,2678000,2,1934.4700,0.025,1933.630000,0.025,0.0,-1.0,49094.333789,-0.000434
4,183663792398270465,BNBUSDT,BUY,1784123022776722,1784123027421000,1784123058214000,4644278,30793000,2,583.9300,0.080,584.540000,0.080,0.0,-1.0,691.373120,0.001044
